# Pruebas — Cola de recordatorios (Medical Remember)
**Requisito:** `cola_recordatorios.py` debe estar en la **misma carpeta** que este notebook.

Ejecuta las celdas en orden. Resultado esperado: `Ran 20 tests ... OK`

In [ ]:
import unittest
from datetime import datetime

from cola_recordatorios import Cola, EstadoToma, GestorTomasDelDia, Toma

DIA = datetime(2026, 10, 2)


def hora(h, m=0):
    return DIA.replace(hour=h, minute=m)


def toma(hid, h, m=0, med="Metformina"):
    return Toma(hid, "Maria Garcia Lopez", med, "850 mg", hora(h, m))


class PruebasCola(unittest.TestCase):
    def test_cola_nueva_esta_vacia(self):
        c = Cola()
        self.assertTrue(c.esta_vacia())
        self.assertEqual(c.tamanio(), 0)

    def test_fifo_primero_en_entrar_primero_en_salir(self):
        c = Cola()
        for x in (1, 2, 3):
            c.encolar(x)
        self.assertEqual([c.desencolar() for _ in range(3)], [1, 2, 3])

    def test_frente_no_quita_el_elemento(self):
        c = Cola()
        c.encolar("a")
        c.encolar("b")
        self.assertEqual(c.frente(), "a")
        self.assertEqual(c.tamanio(), 2)

    def test_desencolar_cola_vacia_lanza_error(self):
        with self.assertRaises(IndexError):
            Cola().desencolar()

    def test_frente_cola_vacia_lanza_error(self):
        with self.assertRaises(IndexError):
            Cola().frente()

    def test_reutilizar_cola_tras_vaciarse(self):
        c = Cola()
        c.encolar(1)
        c.desencolar()
        c.encolar(2)
        self.assertEqual(c.frente(), 2)
        self.assertEqual(c.tamanio(), 1)

    def test_recorrido_no_modifica_la_cola(self):
        c = Cola()
        for x in (1, 2, 3):
            c.encolar(x)
        self.assertEqual(list(c), [1, 2, 3])
        self.assertEqual(len(c), 3)


class PruebasGestor(unittest.TestCase):
    def setUp(self):
        self.g = GestorTomasDelDia()
        # Se entregan desordenadas a propósito
        self.g.cargar_del_dia([toma(3, 20), toma(1, 8), toma(2, 14)])

    def test_carga_ordena_por_hora(self):
        self.assertEqual([t.horario_id for t in self.g.pendientes], [1, 2, 3])

    def test_atender_a_tiempo_marca_tomado(self):
        t = self.g.atender(hora(8, 10))
        self.assertEqual(t.estado, EstadoToma.TOMADO)
        self.assertEqual(len(self.g.pendientes), 2)

    def test_atender_con_mas_de_30_min_marca_tardio(self):
        t = self.g.atender(hora(8, 45))
        self.assertEqual(t.estado, EstadoToma.TARDIO)

    def test_atender_exactamente_30_min_sigue_siendo_tomado(self):
        t = self.g.atender(hora(8, 30))
        self.assertEqual(t.estado, EstadoToma.TOMADO)

    def test_alertas_solo_dentro_de_5_minutos(self):
        self.assertEqual(len(self.g.alertas_proximas(hora(8, 4))), 1)
        self.assertEqual(len(self.g.alertas_proximas(hora(8, 6))), 0)

    def test_alertas_no_modifican_la_cola(self):
        self.g.alertas_proximas(hora(8, 0))
        self.assertEqual(len(self.g.pendientes), 3)

    def test_posponer_reencola_al_final(self):
        self.g.posponer()
        self.assertEqual([t.horario_id for t in self.g.pendientes], [2, 3, 1])
        self.assertEqual(self.g.pendientes.frente().horario_id, 2)

    def test_posponer_incrementa_aplazamientos(self):
        t = self.g.posponer()
        self.assertEqual(t.aplazamientos, 1)

    def test_tercer_aplazamiento_marca_omitido(self):
        g = GestorTomasDelDia()
        g.cargar_del_dia([toma(1, 8)])
        g.posponer()
        g.posponer()
        t = g.posponer()
        self.assertEqual(t.estado, EstadoToma.OMITIDO)
        self.assertTrue(g.pendientes.esta_vacia())
        self.assertEqual(len(g.historial), 1)

    def test_atender_con_cola_vacia_lanza_error(self):
        g = GestorTomasDelDia()
        with self.assertRaises(IndexError):
            g.atender(hora(8))

    def test_siguiente_devuelve_none_si_no_hay_pendientes(self):
        self.assertIsNone(GestorTomasDelDia().siguiente())

    def test_porcentaje_de_cumplimiento(self):
        self.g.atender(hora(8, 5))    # Tomado
        self.g.atender(hora(15, 0))   # Tardío (1 h tarde)
        self.assertAlmostEqual(self.g.porcentaje_cumplimiento(), 50.0)

    def test_porcentaje_sin_historial_es_cero(self):
        self.assertEqual(GestorTomasDelDia().porcentaje_cumplimiento(), 0.0)

In [ ]:
unittest.main(argv=['ignorado'], verbosity=2, exit=False)